Note: in the code (variable names, generated file names and the `halfthroat_id` column) the object is still called *halfthroat*, a legacy name from an earlier version of the method. In the paper it is called a *throat*.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage
import skimage

In [ ]:
filename = 'Leopard'
raw_dimension = 1000

In [ ]:
pores_img = np.fromfile(f"../raw_data/{filename}_pores.raw", dtype=np.int32, sep="")
pores_img = pores_img.reshape([raw_dimension, raw_dimension, raw_dimension])
pores_img_boundaries = np.fromfile(f"../raw_data/{filename}_pore_boundaries.raw", dtype=np.int32, sep="")
pores_img_boundaries = pores_img_boundaries.reshape([raw_dimension, raw_dimension, raw_dimension])

# Here we grow the boundaries, but with a check so that they do not grow into neighbouring pores through a non-tight boundary

In [ ]:
pores_boundaries_numbers = np.load(f'../saved_dicts_and_lists/{filename}_pores_boundaries_numbers.npy', allow_pickle=True).item()
pores_boundaries_numbers = {pores_boundaries_numbers[i]: i for i in pores_boundaries_numbers.keys() if pores_boundaries_numbers[i] >= 0}

In [ ]:
seeds = np.copy(np.abs(pores_img_boundaries))
seeds[np.logical_and(pores_img == -1, seeds == 0)] = -1
seeds_new = np.copy(seeds)

In [ ]:
def grow_seeds():
    global seeds
    ce = np.gradient(seeds)
    for i, j, k in np.transpose(
        np.where(
            np.logical_and(
                skimage.segmentation.find_boundaries(seeds, connectivity=1, mode='inner', background=-1),
                seeds == 0
            )
        )
    ):
        directions = [
            (1, 0, 0), (-1, 0, 0),
            (0, 1, 0), (0, -1, 0),
            (0, 0, 1), (0, 0, -1)
        ]
        for dx, dy, dz in directions:
            ni, nj, nk = i + dx, j + dy, k + dz
            # Check that the new coordinates are within the array bounds
            if 0 <= ni < raw_dimension and 0 <= nj < raw_dimension and 0 <= nk < raw_dimension:
                if seeds[ni, nj, nk] != 0 and seeds[ni, nj, nk] != -1 and pores_img[ni, nj, nk] in pores_boundaries_numbers[seeds[ni, nj, nk]]:
                    seeds_new[i, j, k] = seeds[ni, nj, nk]
    seeds = np.copy(seeds_new)

In [ ]:
%%time
for i in range(3):
    grow_seeds()

# Faster growing, but without checking the pore numbers

Important: the footprint must be exactly this, otherwise the regions may grow into unconnected pores

In [ ]:
footprint = scipy.ndimage.generate_binary_structure(3,1)

In [ ]:
halfthroat_img = np.copy(seeds)
halfthroat_img[halfthroat_img == -1] = 0
solid_indices_mask = pores_img == -1

In [ ]:
%%time
progress0 = len(np.where(np.logical_and(np.logical_not(solid_indices_mask), halfthroat_img == 0))[0]) #for progress control
for i in range(130): # grow regions implementation
    halfthroat_img_old = np.copy(halfthroat_img)
    last_step_mask = halfthroat_img_old != 0
    halfthroat_img = scipy.ndimage.grey_dilation(halfthroat_img, footprint=footprint)
    halfthroat_img[solid_indices_mask] = 0
    halfthroat_img[last_step_mask] = halfthroat_img_old[last_step_mask]
    if i % 10 == 0: #for progress control
        progress = len(np.where(np.logical_and(np.logical_not(solid_indices_mask), halfthroat_img == 0))[0])
        print(f'i: {i}  |  {progress0 - progress}/{progress0}  {int((progress0 - progress)/progress0 * 100)}')

The remaining voids are pores that are not connected to other pores. We do not need them.

In [ ]:
plt.imshow(halfthroat_img[0])

In [ ]:
halfthroat_img.astype(np.int32).tofile(f'../raw_data/{filename}_halfthroats.raw')

Check for lost values (one halfthroat for some reason overlapped another one)

In [ ]:
if sorted(scipy.ndimage.value_indices(halfthroat_img, ignore_value=0).keys()) == sorted([i for i in scipy.ndimage.value_indices(pores_img_boundaries, ignore_value=0).keys() if i > 0]):
    print('No boundary values were lost')
else:
    print('Some boundary values were lost')
    boundary_labels = {i for i in scipy.ndimage.value_indices(pores_img_boundaries, ignore_value=0).keys() if i > 0}
    lost_labels = boundary_labels.difference(scipy.ndimage.value_indices(halfthroat_img, ignore_value=0).keys())
    print(f'{len(lost_labels)} of {len(boundary_labels)} boundary values were lost ({len(lost_labels) / len(boundary_labels) * 100:.2f}%)')